In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

def create_driver():
    from selenium.webdriver.chrome.options import Options
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-dev-shm-usage")
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Initialize
base_url = "https://www.hoosierdata.in.gov/buslookup/page2.aspx?scope=4&geo_area=&name_text=polygon&company_size=Z&datacode=%25"

all_employers = []
start_page = 1
end_page = 2308

driver = create_driver()
wait = WebDriverWait(driver, 10)
driver.get(base_url)

# Set Page Size to 100
try:
    wait.until(EC.presence_of_element_located((By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize")))
    page_size_dropdown = Select(driver.find_element(By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize"))
    page_size_dropdown.select_by_visible_text("100")
    wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
    print("✅ Page size set to 100")
except Exception as e:
    print(f"⚠️ Failed to set page size: {e}")
    driver.quit()
    exit()

# Start scraping
for page_num in range(start_page, end_page + 1):
    print(f"Scraping Page {page_num}...")

    try:
        wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
        table = driver.find_element(By.ID, "resultDetail")
        rows = table.find_elements(By.TAG_NAME, "tr")

        if len(rows) <= 1:
            print(f"⚠️ No data rows found on page {page_num}")
            with open(f"page_{page_num}_debug.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            continue

        for row in rows[1:]:  # Skip header
            cols = row.find_elements(By.TAG_NAME, "td")
            if not cols:
                continue

            employer_data = [col.text.strip() for col in cols]
            if "Details" in employer_data[0]:
                employer_data[0] = employer_data[0].replace("Details", "").strip()

            # Extract details
            phone_number = ""
            contact = ""
            try:
                details_link = row.find_element(By.LINK_TEXT, "Details").get_attribute("href")
                driver.execute_script("window.open(arguments[0]);", details_link)
                driver.switch_to.window(driver.window_handles[1])
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                body_text = driver.find_element(By.TAG_NAME, "body").text

                for line in body_text.split("\n"):
                    if "Phone Number:" in line:
                        phone_number = line.split("Phone Number:")[1].strip()
                    elif "Contact:" in line:
                        contact = line.split("Contact:")[1].strip()

                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            except Exception:
                driver.switch_to.window(driver.window_handles[0])

            combined = employer_data[:6] + [phone_number, contact]
            all_employers.append(combined)

    except Exception as e:
        print(f"⚠️ Error on page {page_num}: {e}")

    # Save every 100 pages
    if page_num % 100 == 0 or page_num == end_page:
        df = pd.DataFrame(all_employers, columns=[
            "Employer Name", "Industry/Business Description (NAICS)", "Address", "City",
            "Employees (#)/Employer Size", "Annual Sales", "Phone Number", "Contact"
        ])
        filename = f"Hoosier_Pages_{start_page}_to_{page_num}.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Saved: {filename}")

    # Go to next page
    try:
        next_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "next")))
        driver.execute_script("arguments[0].click();", next_button)
        wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
    except Exception as e:
        print(f"⚠️ Failed to go to next page from page {page_num}: {e}")
        break

driver.quit()

✅ Page size set to 100
Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...
Scraping Page 8...
Scraping Page 9...
Scraping Page 10...
Scraping Page 11...
Scraping Page 12...
Scraping Page 13...
Scraping Page 14...
Scraping Page 15...
Scraping Page 16...
Scraping Page 17...
Scraping Page 18...
Scraping Page 19...
Scraping Page 20...
Scraping Page 21...
Scraping Page 22...
Scraping Page 23...
Scraping Page 24...
Scraping Page 25...
Scraping Page 26...
Scraping Page 27...
Scraping Page 28...
Scraping Page 29...
Scraping Page 30...
Scraping Page 31...
Scraping Page 32...
Scraping Page 33...
Scraping Page 34...
Scraping Page 35...
Scraping Page 36...
Scraping Page 37...
Scraping Page 38...
Scraping Page 39...
Scraping Page 40...
Scraping Page 41...
Scraping Page 42...
Scraping Page 43...
Scraping Page 44...
Scraping Page 45...
Scraping Page 46...
Scraping Page 47...
Scraping Page 48...
Scraping Page 49...
Scrapi

##### modified trial with updated script from above

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

def create_driver():
    from selenium.webdriver.chrome.options import Options
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-dev-shm-usage")
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def go_to_page(driver, wait, target_page):
    for _ in range(target_page - 1):
        try:
            next_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "next")))
            driver.execute_script("arguments[0].click();", next_button)
            wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
        except Exception as e:
            print(f"⚠️ Failed to reach page {target_page}: {e}")
            return False
    return True

# Initialize
base_url = "https://www.hoosierdata.in.gov/buslookup/page2.aspx?scope=4&geo_area=&name_text=polygon&company_size=Z&datacode=%25"
all_employers = []
start_page = 1
end_page = 2308

driver = create_driver()
wait = WebDriverWait(driver, 10)
driver.get(base_url)

# Set Page Size to 100
try:
    wait.until(EC.presence_of_element_located((By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize")))
    page_size_dropdown = Select(driver.find_element(By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize"))
    page_size_dropdown.select_by_visible_text("100")
    wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
    print("✅ Page size set to 100")
except Exception as e:
    print(f"⚠️ Failed to set page size: {e}")
    driver.quit()
    exit()

# Navigate to start page
if start_page > 1:
    print(f"⏩ Navigating to page {start_page}...")
    if not go_to_page(driver, wait, start_page):
        driver.quit()
        exit()

# Start scraping
for page_num in range(start_page, end_page + 1):
    print(f"🔍 Scraping Page {page_num}...")

    # Verify pagination
    try:
        pagination_text = driver.find_element(By.ID, "ContentPlaceHolder1_lblTopCurrentPage").text
        if f"Now viewing : {page_num} of" not in pagination_text:
            print(f"⚠️ Pagination mismatch on page {page_num}: {pagination_text}")
            with open(f"pagination_mismatch_page_{page_num}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            continue
    except Exception as e:
        print(f"⚠️ Could not verify pagination on page {page_num}: {e}")
        continue

    try:
        wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
        table = driver.find_element(By.ID, "resultDetail")
        rows = table.find_elements(By.TAG_NAME, "tr")

        if len(rows) <= 1:
            print(f"⚠️ No data rows found on page {page_num}")
            with open(f"page_{page_num}_debug.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            continue

        for row in rows[1:]:
            cols = row.find_elements(By.TAG_NAME, "td")
            if not cols:
                continue

            employer_data = [col.text.strip() for col in cols]
            if "Details" in employer_data[0]:
                employer_data[0] = employer_data[0].replace("Details", "").strip()

            all_employers.append(employer_data[:6])

    except Exception as e:
        print(f"⚠️ Error on page {page_num}: {e}")

    # Save every 100 pages or at the end
    if page_num % 100 == 0 or page_num == end_page:
        df = pd.DataFrame(all_employers, columns=[
            "Employer Name", "Industry/Business Description (NAICS)", "Address", "City",
            "Employees (#)/Employer Size", "Annual Sales"
        ])
        filename = f"Hoosier_Pages_{start_page}_to_{page_num}.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Saved: {filename}")

    # Restart browser every 100 pages
    if page_num % 100 == 0:
        driver.quit()
        driver = create_driver()
        wait = WebDriverWait(driver, 10)
        driver.get(base_url)
        try:
            wait.until(EC.presence_of_element_located((By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize")))
            page_size_dropdown = Select(driver.find_element(By.NAME, "ctl00$ContentPlaceHolder1$ddlTopPageSize"))
            page_size_dropdown.select_by_visible_text("100")
            wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
        except:
            print("⚠️ Failed to reset page size after restart.")
            break
        if not go_to_page(driver, wait, page_num + 1):
            break
        continue

    # Go to next page
    try:
        next_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "next")))
        driver.execute_script("arguments[0].click();", next_button)
        wait.until(EC.presence_of_element_located((By.ID, "resultDetail")))
        time.sleep(1)  # Small delay to ensure full load
    except Exception as e:
        print(f"⚠️ Failed to go to next page from page {page_num}: {e}")
        break

driver.quit()

✅ Page size set to 100
🔍 Scraping Page 1...
🔍 Scraping Page 2...
🔍 Scraping Page 3...
🔍 Scraping Page 4...


KeyboardInterrupt: 